# 05 · Hybrid sameDest với chia theo hành vi user ("đặt lại nhiều hơn đặt mới")

Các cohort của notebook 04 được định nghĩa theo **truy vấn hiện tại** (destination đã có trong lịch sử hay chưa), không theo thói quen của user. Notebook này thêm cách chia theo **hành vi** và hỏi liệu nó có vượt tham chiếu đó không, **mà không làm bộ trọng số nào bị nhỏ đi**.

**Công thức và trọng số mỗi ô: không đổi so với notebook 04** (`nbp.hybrid.fusion`): `score = log q_SMLP4Rec + w_p * log q_prior + w_s * log q_sameDest`, `log q = log((1 - alpha) * p + alpha / 100)`, alpha cố định 0.2 (chọn ở notebook 04), lưới `w_p, w_s` trong [0, 3], trọng số mỗi ô theo quy tắc một sai số chuẩn trên valid (SE gom cụm theo user; đơn giản nhất = trọng số lớn nhất nhỏ nhất). Chỉ định nghĩa ô thay đổi.

**Feature hành vi** (`nbp.hybrid.behaviour.history_behaviour`, `cluster_repeat`): `old_share` = tỷ lệ các booking trước đó của user (vị trí 2..L) có cluster đã được đặt sớm hơn trong lịch sử ("old") so với chưa ("new"). Được tính từ lịch sử trước mục tiêu, không bao giờ từ mục tiêu. User chỉ có một booking trước đó (L = 1) không có bằng chứng và tạo thành nhóm riêng.

- **old-leaning**: L >= 2 và `old_share >= cut`; **new-leaning**: L >= 2 và `old_share < cut`; **L = 1**.
- **Ngưỡng** `cut` được chọn trên TRAIN bằng một stump một lần chia trên "mục tiêu là cluster user đã đặt" trong các dòng L >= 2, trong số các ứng viên mà mọi ô kết quả chứa ít nhất **10 % số dòng warm** (sàn độ phủ); được kiểm tra bằng độ ổn định bootstrap cấp user và bằng benchmark kiểm chứng chéo trên valid.
- **Phương án** (1 đến 6 ô) được chấm ngoài fold (CV 2-fold cấp user trên valid, 3 seed). **Một phương án chỉ đủ điều kiện nếu ô nhỏ nhất trên valid chứa >= 10 % số dòng warm** (nên phương án 6 ô được hiển thị nhưng bị loại theo cấu tạo). Trong các phương án đủ điều kiện, quy tắc một-SE chọn phương án ít ô nhất trong một SE của phương án tốt nhất; mức cải thiện so với phương án tham chiếu của notebook 04 (destination known vs new) được báo cáo kèm SE.
- Test đọc một lần, ở cuối. Dòng cold (L = 0) vẫn chỉ dùng prior.

Kernel: `smlp4rec`. Cần `data/interim/recbole/expedia_dest/expedia_dest.inter` và `train.csv` thô.

In [1]:
import functools
import itertools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import summarize, target_rank
from nbp.hybrid.fusion import cluster_se, one_se_pick, same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores
from src.models.smlprec import SMLPREC

logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
RAW = ROOT / "data" / "raw" / "hospitality" / "expedia" / "train.csv"
INTER_DEST = WORK / "expedia_dest" / "expedia_dest.inter"
REF_DIR = ROOT / "results" / "week3_implementation"
RUN_DIR = max((ROOT / "experiments").glob("*_expedia_smlp4rec_week3-repro"))

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6  # prior floor of notebook 03 (cold rows)
TIME_MARGIN = 128  # seconds; float32 timestamp resolution near 1.4e9
RECENCY = 0.7
ALPHAS = [0.05, 0.1, 0.2, 0.3, 0.5]  # uniform-mixing share, one value for all three components
W_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0]  # w_p and w_s, SMLP4Rec fixed at 1
CV_SEEDS = [0, 1, 2]  # user-level 2-fold CV on valid for the cohort schemes
BUCKETS = [("L=1", 1, 1), ("L=2-4", 2, 4), ("L=5-9", 5, 9), ("L>=10", 10, 10**9)]
print("run dir:", RUN_DIR)

run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-02_expedia_smlp4rec_week3-repro


## 1. Dữ liệu, prior và các thành phần điểm (cùng code với notebook 04)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
cluster_ids = np.array(
    [tok2id[str(k)] for k in range(N_CLUSTERS)]
)  # model column of each raw cluster
col_of_model_id = {int(m): k for k, m in enumerate(cluster_ids)}
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]


def split_arrays(loader) -> dict:
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y_model = d[config["ITEM_ID_FIELD"]].numpy()
    q_dest = d["srch_destination_id"].numpy().astype(np.int64)
    h_dest = d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64)
    pos = np.arange(seq.shape[1])[None, :] < length[:, None]
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": np.array([col_of_model_id[int(t)] for t in y_model]),  # raw cluster index 0..99
        "user": d[config["USER_ID_FIELD"]].numpy(),
        "old": ((seq == y_model[:, None]) & pos).any(1),
        "q_dest": q_dest,
        "h_dest": h_dest,
        "q_seen": ((h_dest == q_dest[:, None]) & pos).any(1),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
bucket_masks = {
    sp: {b: (a["length"] >= lo) & (a["length"] <= hi) for b, lo, hi in BUCKETS}
    for sp, a in arr.items()
}
print({sp: len(a["y"]) for sp, a in arr.items()}, f"{time.time() - t0:.0f}s")

{'valid': 218670, 'test': 218670} 48s


In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_seconds: float) -> dict:
    """Cluster count tables from bookings with unix time < cut_seconds (one chunked pass over train.csv)."""
    cols = ["date_time", "is_booking", "srch_destination_id", "hotel_market", "hotel_cluster"]
    dtypes = {c: "int32" for c in cols if c != "date_time"}
    parts = []
    for ch in pd.read_csv(RAW, usecols=cols, chunksize=2_000_000, dtype=dtypes):
        ch = ch[ch.is_booking == 1]
        ts = (pd.to_datetime(ch["date_time"]) - pd.Timestamp("1970-01-01")) // pd.Timedelta(
            seconds=1
        )
        parts.append(
            ch.loc[
                ts.values < cut_seconds, ["srch_destination_id", "hotel_market", "hotel_cluster"]
            ]
        )
    b = pd.concat(parts, ignore_index=True)

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(
            zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))
        ),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_prob(dests: np.ndarray) -> np.ndarray:
    """(n, 100) prior distribution of each query destination, raw cluster order."""
    uniq, inv = np.unique(dests, return_inverse=True)
    return np.stack([prior_vector(int(d)) for d in uniq]).astype(np.float32)[inv]


t1 = time.time()
tables = build_prior_tables(cut)
print(
    f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s"
)

model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_prob(a: dict, batch: int = 8192) -> np.ndarray:
    """(n, 100) softmax of the model over the 100 clusters (padding column excluded), raw cluster order."""
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {
                model.ITEM_SEQ: a["seq_t"][i : i + batch],
                model.ITEM_SEQ_LEN: a["length_t"][i : i + batch],
            }
        )
        s[:, 0] = float("-inf")
        out.append(torch.softmax(s, dim=1).numpy()[:, cluster_ids])
    return np.concatenate(out)


p_model, p_prior, sd = {}, {}, {}
for sp, a in arr.items():
    p_model[sp], p_prior[sp] = model_prob(a), prior_prob(a["q_dest"])
    sd[sp] = same_dest_scores(
        a["seq"],
        a["length"],
        a["h_dest"],
        a["q_dest"],
        dataset.item_num,
        base=RECENCY,
        normalize=True,
    )[:, cluster_ids]
print("components ready", {sp: p_model[sp].shape for sp in arr})

prior cut (unix s) 1412776576; 2463082 bookings, 34506 destinations; 30s


components ready {'valid': (218670, 100), 'test': (218670, 100)}


In [4]:
CUTS = [0.25, 1 / 3, 0.4, 0.5, 0.6]  # old-leaning: old_share >= cut
FLOOR = 0.10  # every weight set must cover at least this share of the warm rows
N_BOOT_CUT = 50
ALPHA = 0.2  # chosen in notebook 04

from nbp.hybrid.behaviour import history_behaviour, split_gain

t5 = time.time()
tr_arr = split_arrays(train_data)
for a in (tr_arr, arr["valid"], arr["test"]):
    a["old_share"] = history_behaviour(a["seq"], a["length"], a["h_dest"])["cluster_repeat"]
print(
    f"train {len(tr_arr['y'])} rows; behaviour features for train/valid/test {time.time() - t5:.0f}s"
)


def rank(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    return target_rank(scores, y)


def joint(r: np.ndarray) -> np.ndarray:
    """Per-row Recall@5 + NDCG@5 (float32)."""
    r = r.astype(np.float32)
    return ((r <= 5) * (1.0 + 1.0 / np.log2(r + 1))).astype(np.float32)


def complexity(c: tuple) -> tuple:
    """Simpler = smaller largest weight, then smaller weight sum, then more smoothing (as notebook 04)."""
    return (max(c[1], c[2]), c[1] + c[2], -c[0])


def hybrid_terms(sp: str, alpha: float) -> tuple:
    return (
        smoothed_log(p_model[sp], alpha),
        smoothed_log(p_prior[sp], alpha),
        smoothed_log(same_dest_distribution(sd[sp]), alpha),
    )


t6 = time.time()
terms = {sp: hybrid_terms(sp, ALPHA) for sp in arr}
GRID = list(itertools.product(W_GRID, W_GRID))
grid_rank = {
    sp: {
        c: rank(terms[sp][0] + c[0] * terms[sp][1] + c[1] * terms[sp][2], arr[sp]["y"]).astype(
            np.int16
        )
        for c in GRID
    }
    for sp in arr
}
print(
    f"{len(GRID)} (w_p, w_s) pairs ranked on valid and test (alpha {ALPHA}): {time.time() - t6:.0f}s"
)

train 1749368 rows; behaviour features for train/valid/test 6s


100 (w_p, w_s) pairs ranked on valid and test (alpha 0.2): 41s


## 2. Feature hành vi: nó phân tách điều gì, và ngưỡng (chỉ train)

Kết quả dùng cho stump: mục tiêu có phải cluster user **đã đặt** (`old`) hay không. Các cột hồ sơ: `tgt_old`, `known_dest` (destination truy vấn đã có trong lịch sử), và tỷ lệ của các dòng trên số dòng warm.

In [5]:
w2 = tr_arr["length"] >= 2
share_l2 = float(w2.mean())
x_tr, y_tr = tr_arr["old_share"][w2], tr_arr["old"][w2].astype(np.float64)

BINS = [0, 1e-9, 0.25, 1 / 3 - 1e-9, 0.5 - 1e-9, 0.6, 0.8, 1 - 1e-9, 1 + 1e-9]
LABELS = ["0", "(0,0.25)", "[0.25,0.33)", "[0.33,0.5)", "[0.5,0.6)", "[0.6,0.8)", "[0.8,1)", "1"]
prof = {}
for lab, lo, hi in zip(LABELS, BINS[:-1], BINS[1:]):
    m = w2 & (tr_arr["old_share"] >= lo) & (tr_arr["old_share"] < hi)
    prof[lab] = {
        "share_of_warm_rows": round(float(m.mean()), 4),
        "tgt_old": round(float(tr_arr["old"][m].mean()), 4),
        "known_dest": round(float(tr_arr["q_seen"][m].mean()), 4),
    }
m1 = tr_arr["length"] == 1
prof["L=1 (no history to measure)"] = {
    "share_of_warm_rows": round(float(m1.mean()), 4),
    "tgt_old": round(float(tr_arr["old"][m1].mean()), 4),
    "known_dest": round(float(tr_arr["q_seen"][m1].mean()), 4),
}
print("train: old_share profile")
display(pd.DataFrame(prof).T)


def stump_table(weight: np.ndarray | None = None) -> dict:
    return split_gain(
        x_tr, y_tr, CUTS, above_or_equal=True, weight=None if weight is None else weight[w2]
    )


def pick_cut(g: dict) -> float:
    """Best stump gain among cuts whose new-leaning and old-leaning cells both reach the coverage floor."""
    ok = {
        c: v
        for c, v in g.items()
        if v["share_high"] * share_l2 >= FLOOR
        and (1 - v["share_high"]) * share_l2 >= FLOOR
        and v["gain"] > 0
    }
    return max(ok, key=lambda c: ok[c]["gain"])


g_train = stump_table()
CUT = pick_cut(g_train)
stump_tab = {
    round(c, 4): {
        "gain": round(v["gain"], 1),
        "old_leaning_share_of_warm": round(v["share_high"] * share_l2, 4),
        "new_leaning_share_of_warm": round((1 - v["share_high"]) * share_l2, 4),
        "tgt_old_if_below": round(v["mean_low"], 4),
        "tgt_old_if_at_or_above": round(v["mean_high"], 4),
        "meets_floor": bool(
            v["share_high"] * share_l2 >= FLOOR and (1 - v["share_high"]) * share_l2 >= FLOOR
        ),
    }
    for c, v in g_train.items()
}
print("stump on train (outcome: target already booked by the user, rows with L >= 2)")
display(pd.DataFrame(stump_tab).T)
print(f"chosen cut on train: {CUT:.4g}")

u_tr, inv_tr = np.unique(tr_arr["user"], return_inverse=True)
rng = np.random.default_rng(0)
boot_picks = []
for _ in range(N_BOOT_CUT):
    cnt = np.bincount(rng.integers(0, len(u_tr), len(u_tr)), minlength=len(u_tr))
    boot_picks.append(pick_cut(stump_table(cnt[inv_tr].astype(np.float64))))
stability = {
    str(round(float(c), 4)): round(boot_picks.count(c) / N_BOOT_CUT, 4)
    for c in sorted(set(boot_picks))
}
print(f"bootstrap stability of the cut ({N_BOOT_CUT} user-level resamples of train): {stability}")

train: old_share profile


,share_of_warm_rows,tgt_old,known_dest
0,0.3171,0.2119,0.2141
"(0,0.25)",0.0967,0.3543,0.2905
"[0.25,0.33)",0.0757,0.3844,0.3289
"[0.33,0.5)",0.1250,0.4250,0.3812
"[0.5,0.6)",0.0743,0.4115,0.4040
"[0.6,0.8)",0.0376,0.5461,0.5515
"[0.8,1)",0.0057,0.7069,0.7479
1,0.0281,0.3077,0.3432
L=1 (no history to measure),0.2397,0.1355,0.1617


stump on train (outcome: target already booked by the user, rows with L >= 2)


,gain,old_leaning_share_of_warm,new_leaning_share_of_warm,tgt_old_if_below,tgt_old_if_at_or_above,meets_floor
0.2500,10263.1,0.3464,0.4139,0.2452,0.4215,True
0.3333,8324.4,0.2707,0.4896,0.2667,0.4319,True
0.4000,7302.6,0.1989,0.5614,0.2814,0.45,True
0.5000,3977.1,0.1457,0.6146,0.2989,0.4378,True
0.6000,2691.9,0.0714,0.6889,0.311,0.4653,False


chosen cut on train: 0.25


bootstrap stability of the cut (50 user-level resamples of train): {'0.25': 1.0}


## 3. Định nghĩa ô, độ phủ, và trọng số mỗi ô (valid)

`pick_cell` là quy tắc của notebook 04: trong 100 cặp (w_p, w_s) (alpha 0.2), cặp đơn giản nhất nằm trong một SE gom cụm theo user của Recall@5 + NDCG@5 tốt nhất trên valid; ở các ô không có dòng known-destination nào thì `w_s` cố định 0 (số hạng là hằng số ở đó).

In [6]:
yv, uv, qs_v = arr["valid"]["y"], arr["valid"]["user"], arr["valid"]["q_seen"]


def behaviour_cells(a: dict, cut: float) -> np.ndarray:
    """0 = L=1 (no history to measure), 1 = new-leaning, 2 = old-leaning."""
    return np.where(a["length"] == 1, 0, np.where(a["old_share"] >= cut, 2, 1))


def schemes_for(a: dict, cut: float) -> dict:
    qs = a["q_seen"].astype(int)
    b3 = behaviour_cells(a, cut)
    b2 = (b3 == 2).astype(int)
    return {
        "global (1 set)": np.zeros(len(qs), int),
        "known vs new destination (2) [notebook 04]": qs,
        "old-leaning vs rest (2)": b2,
        "L=1 | new-leaning | old-leaning (3)": b3,
        "known/new destination x old-leaning/rest (4)": qs * 2 + b2,
        "known/new destination x (L=1 | new | old) (6)": qs * 3 + b3,
    }


REF = "known vs new destination (2) [notebook 04]"
BEH_SCHEMES = [n for n in schemes_for(arr["valid"], CUT) if "old-leaning" in n or "old)" in n]
CELL_LABEL = {
    "global (1 set)": {0: "all warm rows"},
    REF: {0: "destination new to the user", 1: "destination already in history"},
    "old-leaning vs rest (2)": {0: "L=1 or new-leaning", 1: "old-leaning"},
    "L=1 | new-leaning | old-leaning (3)": {0: "L=1", 1: "new-leaning", 2: "old-leaning"},
    "known/new destination x old-leaning/rest (4)": {
        0: "new dest, rest",
        1: "new dest, old-leaning",
        2: "known dest, rest",
        3: "known dest, old-leaning",
    },
    "known/new destination x (L=1 | new | old) (6)": {
        0: "new dest, L=1",
        1: "new dest, new-leaning",
        2: "new dest, old-leaning",
        3: "known dest, L=1",
        4: "known dest, new-leaning",
        5: "known dest, old-leaning",
    },
}
cfg = list(GRID)


def pick_cell(mask: np.ndarray) -> tuple:
    cand = (
        cfg if qs_v[mask].any() else [c for c in cfg if c[1] == 0.0]
    )  # w_s is irrelevant without known-destination rows
    return one_se_pick(
        {c: joint(grid_rank["valid"][c][mask]) for c in cand}, uv[mask], lambda c: (max(c), sum(c))
    )[0]


SCHEMES = schemes_for(arr["valid"], CUT)
n_cells = {n: len(np.unique(c)) for n, c in SCHEMES.items()}
cover = {n: np.bincount(c, minlength=n_cells[n]) / len(c) for n, c in SCHEMES.items()}
eligible = {n: bool(cover[n].min() >= FLOOR) for n in SCHEMES}
print("valid coverage per cell (share of the warm rows) and eligibility (smallest cell >= 10 %):")
for n in SCHEMES:
    print(f"  {n:48s} {np.round(cover[n], 3).tolist()}  eligible: {eligible[n]}")

valid coverage per cell (share of the warm rows) and eligibility (smallest cell >= 10 %):
  global (1 set)                                   [1.0]  eligible: True
  known vs new destination (2) [notebook 04]       [0.7, 0.3]  eligible: True
  old-leaning vs rest (2)                          [0.596, 0.404]  eligible: True
  L=1 | new-leaning | old-leaning (3)              [0.178, 0.418, 0.404]  eligible: True
  known/new destination x old-leaning/rest (4)     [0.465, 0.235, 0.131, 0.169]  eligible: True
  known/new destination x (L=1 | new | old) (6)    [0.151, 0.314, 0.235, 0.028, 0.104, 0.169]  eligible: False


## 4. Benchmark ngoài fold: phương án nào, và cách chia theo hành vi có thêm gì không?

Mỗi phương án: tinh chỉnh trọng số các ô trên một nửa user của valid, chấm nửa còn lại, cả hai chiều, ba seed chia user. Điểm là trung bình Recall@5 + NDCG@5 theo từng dòng. Quy tắc một-SE sau đó chọn, trong các phương án **đủ điều kiện**, phương án có ít ô nhất trong một SE của phương án tốt nhất.

In [7]:
CV_SEEDS = [0, 1, 2]
u_va, inv_va = np.unique(uv, return_inverse=True)
folds = {s: np.random.default_rng(s).integers(0, 2, len(u_va))[inv_va] for s in CV_SEEDS}


def oof_joint(cell_of_row: np.ndarray) -> np.ndarray:
    """Out-of-fold per-row joint score, averaged over the fold seeds."""
    per_seed = []
    for s in CV_SEEDS:
        r_oof = np.zeros(len(yv), np.int64)
        for half in (0, 1):
            tune, ev = folds[s] != half, folds[s] == half
            for c_id in np.unique(cell_of_row):
                m = cell_of_row == c_id
                r_oof[m & ev] = grid_rank["valid"][pick_cell(m & tune)][m & ev]
        per_seed.append(joint(r_oof))
    return np.mean(per_seed, axis=0)


t7 = time.time()
oof = {n: oof_joint(c) for n, c in SCHEMES.items()}
print(f"out-of-fold CV over {len(SCHEMES)} schemes: {time.time() - t7:.0f}s")

elig = {n: oof[n] for n in SCHEMES if eligible[n]}
SCHEME, BEST_ELIG, ok_schemes = one_se_pick(elig, uv, lambda n: (n_cells[n],))
best_beh = max([n for n in BEH_SCHEMES if eligible[n]], key=lambda n: float(oof[n].mean()))
scheme_table = {}
for n in SCHEMES:
    d_ref = oof[n] - oof[REF]
    d_glob = oof[n] - oof["global (1 set)"]
    scheme_table[n] = {
        "cells": n_cells[n],
        "min_cell_share": round(float(cover[n].min()), 4),
        "eligible": eligible[n],
        "oof_joint": round(float(oof[n].mean()), 5),
        "gain_vs_global": f"{d_glob.mean():+.5f} (SE {cluster_se(d_glob, uv):.5f})",
        "gain_vs_reference": f"{d_ref.mean():+.5f} (SE {cluster_se(d_ref, uv):.5f})",
        "within_one_se_of_best_eligible": n in ok_schemes,
    }
display(pd.DataFrame(scheme_table).T)
print(f"best eligible scheme out of fold: {BEST_ELIG}")
print(f"one-SE pick among eligible (fewest cells): {SCHEME}")
print(f"best eligible behaviour scheme: {best_beh}")

# the cut itself: vary it one at a time on the 2-cell behaviour scheme (all cuts reported, floor flagged)
cut_cv = {}
for c in CUTS:
    cell = (behaviour_cells(arr["valid"], c) == 2).astype(int)
    o = oof_joint(cell)
    d_ref, d_glob = o - oof[REF], o - oof["global (1 set)"]
    cut_cv[round(c, 4)] = {
        "old_leaning_share": round(float(cell.mean()), 4),
        "meets_floor": bool(min(cell.mean(), 1 - cell.mean()) >= FLOOR),
        "oof_joint": round(float(o.mean()), 5),
        "gain_vs_global": f"{d_glob.mean():+.5f} (SE {cluster_se(d_glob, uv):.5f})",
        "gain_vs_reference": f"{d_ref.mean():+.5f} (SE {cluster_se(d_ref, uv):.5f})",
    }
print(
    "\nvarying the cut on the 'old-leaning vs rest (2)' scheme (train choice:", round(CUT, 4), ")"
)
display(pd.DataFrame(cut_cv).T)

out-of-fold CV over 6 schemes: 25s


,cells,min_cell_share,eligible,oof_joint,gain_vs_global,gain_vs_reference,within_one_se_of_best_eligible
global (1 set),1,1.0,True,1.07654,+0.00000 (SE 0.00000),-0.00029 (SE 0.00024),False
known vs new destination (2) [notebook 04],2,0.2999,True,1.07683,+0.00029 (SE 0.00024),+0.00000 (SE 0.00000),True
old-leaning vs rest (2),2,0.4037,True,1.07623,-0.00031 (SE 0.00032),-0.00060 (SE 0.00034),False
L=1 | new-leaning | old-leaning (3),3,0.1783,True,1.0757,-0.00084 (SE 0.00037),-0.00112 (SE 0.00037),False
known/new destination x old-leaning/rest (4),4,0.1313,True,1.07646,-0.00008 (SE 0.00033),-0.00036 (SE 0.00021),False
known/new destination x (L=1 | new | old) (6),6,0.0276,False,1.07625,-0.00029 (SE 0.00035),-0.00058 (SE 0.00024),False


best eligible scheme out of fold: known vs new destination (2) [notebook 04]
one-SE pick among eligible (fewest cells): known vs new destination (2) [notebook 04]
best eligible behaviour scheme: known/new destination x old-leaning/rest (4)



varying the cut on the 'old-leaning vs rest (2)' scheme (train choice: 0.25 )


,old_leaning_share,meets_floor,oof_joint,gain_vs_global,gain_vs_reference
0.2500,0.4037,True,1.07623,-0.00031 (SE 0.00032),-0.00060 (SE 0.00034)
0.3333,0.3065,True,1.07655,+0.00001 (SE 0.00027),-0.00027 (SE 0.00031)
0.4000,0.2218,True,1.07649,-0.00005 (SE 0.00020),-0.00033 (SE 0.00027)
0.5000,0.1519,True,1.07639,-0.00015 (SE 0.00022),-0.00043 (SE 0.00030)
0.6000,0.0717,False,1.07633,-0.00021 (SE 0.00011),-0.00049 (SE 0.00025)


## 5. Trọng số cuối cùng mỗi ô (toàn bộ tập valid), rồi đọc test một lần

In [8]:
EVAL = {
    "reference (notebook 04)": REF,
    "chosen by the rule": SCHEME,
    "best behaviour scheme": best_beh,
}
final_w = {}
for n in dict.fromkeys(EVAL.values()):
    final_w[n] = {int(c): pick_cell(SCHEMES[n] == c) for c in np.unique(SCHEMES[n])}
SCHEMES_T = schemes_for(arr["test"], CUT)
yt = arr["test"]["y"]
M_t, P_t, S_t = terms["test"]


def test_rank(name: str) -> np.ndarray:
    cid, w = SCHEMES_T[name], final_w[name]
    wp = np.array([w[int(c)][0] for c in cid], np.float32)[:, None]
    ws = np.array([w[int(c)][1] for c in cid], np.float32)[:, None]
    return rank(M_t + wp * P_t + ws * S_t, yt)


rows = []
for name, w in final_w.items():
    for c_id, (wp, ws) in w.items():
        rows.append(
            {
                "scheme": name,
                "cell": CELL_LABEL[name][c_id],
                "valid_share": round(float((SCHEMES[name] == c_id).mean()), 4),
                "test_share": round(float((SCHEMES_T[name] == c_id).mean()), 4),
                "w_p": wp,
                "w_s": ws if qs_v[SCHEMES[name] == c_id].any() else "n/a (constant)",
            }
        )
display(pd.DataFrame(rows))

ranks_t = {label: test_rank(name) for label, name in EVAL.items()}
ref04 = json.loads((REF_DIR / "smlprec_expedia_hybrid_samedest.json").read_text(encoding="utf-8"))[
    "benchmark"
]["test"]["warm"]
got = summarize(ranks_t["reference (notebook 04)"])
want = ref04["SMLP4Rec + prior + sameDest (this notebook)"]
print(
    f"reference reproduces notebook 04: Recall@5 {got['recall@5']} vs {want['recall@5']}, NDCG@5 {got['ndcg@5']} vs {want['ndcg@5']}"
)
assert got["recall@5"] == want["recall@5"] and got["ndcg@5"] == want["ndcg@5"]

,scheme,cell,valid_share,test_share,w_p,w_s
0,known vs new destination (2) [notebook 04],destination new to the user,0.7001,0.7009,2.00,n/a (constant)
1,known vs new destination (2) [notebook 04],destination already in history,0.2999,0.2991,1.25,1.0
2,known/new destination x old-leaning/rest (4),"new dest, rest",0.4650,0.4717,2.00,n/a (constant)
3,known/new destination x old-leaning/rest (4),"new dest, old-leaning",0.2351,0.2292,1.50,n/a (constant)
4,known/new destination x old-leaning/rest (4),"known dest, rest",0.1313,0.1336,1.25,1.25
5,known/new destination x old-leaning/rest (4),"known dest, old-leaning",0.1686,0.1655,1.25,1.0


reference reproduces notebook 04: Recall@5 0.6106 vs 0.6106, NDCG@5 0.4539 vs 0.4539


In [9]:
inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(
    ["user_id:token", "timestamp:float"], kind="stable"
)
first = inter.drop_duplicates("user_id:token", keep="first")
f_test = first[first["timestamp:float"] >= t_valid_max]
cold_test = rank(
    np.log(prior_prob(f_test["srch_destination_id:float"].values.astype(np.int64)) + EPS),
    f_test["item_id:token"].astype(int).values,
)

COLS = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
bench = {"warm": {}, "all_events": {}}
for label, r in ranks_t.items():
    bench["warm"][label] = summarize(r)
    bench["all_events"][label] = summarize(np.concatenate([r, cold_test]))
for scope in ("warm", "all_events"):
    print(f"TEST, {scope}")
    display(pd.DataFrame(bench[scope]).T[COLS])

boot = {}
for label in ("chosen by the rule", "best behaviour scheme"):
    ra, rb = ranks_t[label], ranks_t["reference (notebook 04)"]
    boot[f"{label} minus reference"] = {
        "warm": paired_bootstrap(ra, rb, k=5),
        "all_events": paired_bootstrap(
            np.concatenate([ra, cold_test]), np.concatenate([rb, cold_test]), k=5
        ),
    }
print("paired bootstrap, test (1000 resamples, seed 0, 95% CI)")
display(
    pd.DataFrame(
        {
            (k, scope): {
                m: f"{v[scope][m]['diff']:+.4f} [{v[scope][m]['ci95'][0]:+.4f}, {v[scope][m]['ci95'][1]:+.4f}]"
                for m in ("recall@5", "ndcg@5")
            }
            for k, v in boot.items()
            for scope in v
        }
    ).T
)


def r5_n5(r: np.ndarray, m: np.ndarray) -> str:
    v = summarize(r, m)
    return f"{v['recall@5']:.4f} / {v['ndcg@5']:.4f}"


at = arr["test"]
b3_t = behaviour_cells(at, CUT)
slices = {
    "L=1": b3_t == 0,
    "new-leaning (L>=2)": b3_t == 1,
    "old-leaning (L>=2)": b3_t == 2,
    "known destination": at["q_seen"],
    "new destination": ~at["q_seen"],
    "target already booked": at["old"],
    "target new to the user": ~at["old"],
}
slice_table = {
    s: {lab: r5_n5(r, m) for lab, r in ranks_t.items()} | {"rows": int(m.sum())}
    for s, m in slices.items()
}
print("TEST slices, Recall@5 / NDCG@5")
display(pd.DataFrame(slice_table).T)

TEST, warm


,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
reference (notebook 04),0.6106,0.4539,0.7741,0.5070,0.9090,0.5413
chosen by the rule,0.6106,0.4539,0.7741,0.5070,0.9090,0.5413
best behaviour scheme,0.6110,0.4539,0.7745,0.5069,0.9089,0.5412


TEST, all_events


,recall@5,ndcg@5,recall@10,ndcg@10,recall@20,ndcg@20
reference (notebook 04),0.5925,0.4349,0.7608,0.4894,0.9027,0.5255
chosen by the rule,0.5925,0.4349,0.7608,0.4894,0.9027,0.5255
best behaviour scheme,0.5927,0.4349,0.7610,0.4894,0.9026,0.5254


paired bootstrap, test (1000 resamples, seed 0, 95% CI)


recall@5  \
chosen by the rule minus reference    warm        +0.0000 [+0.0000, +0.0000]   
                                      all_events  +0.0000 [+0.0000, +0.0000]   
best behaviour scheme minus reference warm        +0.0003 [-0.0000, +0.0007]   
                                      all_events  +0.0003 [-0.0000, +0.0006]   

                                                                      ndcg@5  
chosen by the rule minus reference    warm        +0.0000 [+0.0000, +0.0000]  
                                      all_events  +0.0000 [+0.0000, +0.0000]  
best behaviour scheme minus reference warm        -0.0000 [-0.0002, +0.0002]  
                                      all_events  -0.0000 [-0.0002, +0.0002]

TEST slices, Recall@5 / NDCG@5


,reference (notebook 04),chosen by the rule,best behaviour scheme,rows
L=1,0.5659 / 0.4082,0.5659 / 0.4082,0.5659 / 0.4082,39993
new-leaning (L>=2),0.5845 / 0.4236,0.5845 / 0.4236,0.5845 / 0.4237,92367
old-leaning (L>=2),0.6593 / 0.5076,0.6593 / 0.5076,0.6601 / 0.5075,86310
known destination,0.7401 / 0.6175,0.7401 / 0.6175,0.7401 / 0.6176,65407
new destination,0.5554 / 0.3842,0.5554 / 0.3842,0.5559 / 0.3841,153263
target already booked,0.8757 / 0.7572,0.8757 / 0.7572,0.8809 / 0.7631,67216
target new to the user,0.4930 / 0.3194,0.4930 / 0.3194,0.4912 / 0.3167,151454


## 6. Lưu

In [10]:
result = {
    "formula": "notebook 04 sameDest hybrid, only the cell definition (which (w_p, w_s) applies) changes; alpha fixed at 0.2; cold rows prior only",
    "behaviour_feature": "old_share = share of past bookings (positions 2..L) whose cluster was already booked earlier in the history (nbp.hybrid.behaviour.history_behaviour cluster_repeat); L=1 has no value and forms its own group",
    "cut": {
        "chosen_on_train": CUT,
        "rule": "old-leaning = L >= 2 and old_share >= cut; best stump gain on 'target already booked' among cuts whose cells all reach the floor",
        "floor_share_of_warm_rows": FLOOR,
        "stump_on_train": {str(k): v for k, v in stump_tab.items()},
        "bootstrap_stability": stability,
        "profile_train": prof,
        "cv_varying_cut": {str(k): v for k, v in cut_cv.items()},
    },
    "schemes": scheme_table,
    "chosen_scheme_fewest_cells_within_one_se": SCHEME,
    "best_behaviour_scheme": best_beh,
    "final_weights_per_cell": {
        n: {CELL_LABEL[n][c]: list(w) for c, w in ws.items()} for n, ws in final_w.items()
    },
    "benchmark_test": bench,
    "test_slices_recall@5_ndcg@5": slice_table,
    "bootstrap_test_95ci": boot,
}
for path in (
    RUN_DIR / "hybrid_behaviour_split.json",
    REF_DIR / "smlprec_expedia_hybrid_behaviour_split.json",
):
    path.write_text(json.dumps(result, indent=2, default=float), encoding="utf-8")
print("saved", REF_DIR / "smlprec_expedia_hybrid_behaviour_split.json")

saved C:\workspace\PROJECTS\next-best-product-recommendation\results\week3_implementation\smlprec_expedia_hybrid_behaviour_split.json
